<a href="https://colab.research.google.com/github/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA/blob/main/notebooks/30_expansion/30_expansion_mapbiomas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 30 · Expansión agropecuaria por celda de 2 km

**Autor:** Daniel

**Propósito:** calcular en cada celda de la grilla común el % del área en uso agropecuario en 2000 y en 2024, y su cambio (capa de expansión de la tabla 14.4 de la ruta de trabajo).

**Entradas:**
- `datos/comunes/grilla_2km.gpkg` (cuaderno 02, DVC).
- `datos/comunes/altillanura_municipios.gpkg` (cuaderno 01, DVC), solo para el mapa.
- Mapas de cobertura de MapBiomas Colombia, Colección 3, un GeoTIFF por municipio:
  - 2024: `modulos/expansion/mapas_tif/` (DVC).
  - 2000: `modulos/expansion/mapas_tif_2000/` (DVC).
- Tablas municipales de MapBiomas: `modulos/expansion/datos/mapbiomas_crudos/` (Git), para verificar.

**Salidas:**
- `datos/comunes/agro_celda.csv` (DVC): una fila por celda.
- `modulos/expansion/graficas/30_agro_celda.png`.

**Método:**
- Uso agropecuario = clases 9 (plantación forestal), 21 (mosaico de agricultura y pastos) y 35 (palma de aceite). Son las tres clases del nivel 1 "Zona agrícola y ganadera" que aparecen en la Altillanura, las mismas de las series municipales.
- Cada píxel de 30 m se pasa a EPSG:9377 y se asigna a la celda de 2 km donde cae su centro.
- El % agropecuario de una celda es el área agropecuaria sobre el área con dato de esa celda. El cambio se da en puntos porcentuales.
- `frac_valida` dice qué parte de la celda tiene dato en el mapa. Las celdas con menos de la mitad del área con dato quedan con `celda_valida = False`.

**Columnas de `agro_celda.csv`** (nombres acordados con el cuaderno 61):

| Columna | Qué es |
|---|---|
| `id_celda` | Llave de la grilla común |
| `agro_pct_2000`, `agro_pct_2024` | % del área con dato en uso agropecuario |
| `cambio_agro_pp` | `agro_pct_2024 − agro_pct_2000`, en puntos porcentuales |
| `agro_ha_2000`, `agro_ha_2024` | Hectáreas en uso agropecuario |
| `frac_valida_2000`, `frac_valida_2024` | Fracción del área de la celda con dato en el mapa |
| `celda_valida` | `True` si los dos años tienen dato en al menos la mitad de la celda |

## Paso 1. Rutas y librerías
- En **Colab** clona el repo en `/content/Proyecto-ALTILLANURA`.
- En **local** toma como raíz la carpeta del repo (dos niveles arriba del cuaderno).
- `traer(ruta)` baja de DVC un archivo o carpeta cuando hace falta.

In [ ]:
import os, sys, re, glob, unicodedata, subprocess
EN_COLAB = 'google.colab' in sys.modules
REPO = 'https://github.com/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA.git'

if EN_COLAB:
    RAIZ = '/content/Proyecto-ALTILLANURA'
    if not os.path.exists(RAIZ):
        subprocess.run(['git', 'clone', '-q', REPO, RAIZ], check=True)
else:
    RAIZ = os.path.abspath(os.path.join('..', '..'))

COMUNES = os.path.join(RAIZ, 'datos', 'comunes')
EXPANSION = os.path.join(RAIZ, 'modulos', 'expansion')
CRUDOS = os.path.join(EXPANSION, 'datos', 'mapbiomas_crudos')
GRAFICAS = os.path.join(EXPANSION, 'graficas')
os.makedirs(GRAFICAS, exist_ok=True)

def traer(ruta):
    if os.path.exists(os.path.join(RAIZ, ruta)) and (not os.path.isdir(os.path.join(RAIZ, ruta)) or os.listdir(os.path.join(RAIZ, ruta))):
        return
    if EN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dvc[gs]'], check=True)
    subprocess.run([sys.executable, '-m', 'dvc', 'pull', '-q', ruta], cwd=RAIZ, check=True)
    print('Bajado de DVC:', ruta)

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from pyproj import Transformer
import matplotlib.pyplot as plt

print('Raíz del repo:', RAIZ)

## Paso 2. Grilla y mapas de cobertura
Cada mapa se reconoce por el nombre del municipio (o su código DIVIPOLA) en el nombre del archivo. Si el nombre trae un año, tiene que coincidir con el de la carpeta.

In [ ]:
traer('datos/comunes/grilla_2km.gpkg')
traer('datos/comunes/altillanura_municipios.gpkg')
traer('modulos/expansion/mapas_tif')
traer('modulos/expansion/mapas_tif_2000')

grilla = gpd.read_file(os.path.join(COMUNES, 'grilla_2km.gpkg')).to_crs(9377)
mpios = gpd.read_file(os.path.join(COMUNES, 'altillanura_municipios.gpkg')).to_crs(9377)
mpios['mpio_cdpmp'] = mpios['mpio_cdpmp'].astype(str).str.zfill(5)

CARPETAS = {2000: os.path.join(EXPANSION, 'mapas_tif_2000'), 2024: os.path.join(EXPANSION, 'mapas_tif')}
ANIOS = sorted(CARPETAS)

def normalizar(texto):
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', texto.lower()).strip('_')

NOMBRES = {cod: normalizar(nombre) for cod, nombre in zip(mpios['mpio_cdpmp'], mpios['mpio_cnmbr'])}

def municipio_de(ruta):
    base = os.path.splitext(os.path.basename(ruta))[0]
    separado = normalizar(re.sub(r'(?<=[a-záéíóúñ])(?=[A-ZÁÉÍÓÚÑ])', '_', base))
    for cod, nombre in NOMBRES.items():
        if cod in separado or nombre.replace('_', '') in separado.replace('_', ''):
            return cod
    return None

archivos = {}
for anio, carpeta in CARPETAS.items():
    rutas = sorted(glob.glob(os.path.join(carpeta, '*.tif')))
    archivos[anio] = {}
    for ruta in rutas:
        cod = municipio_de(ruta)
        if cod is None:
            raise ValueError(f'No se reconoce el municipio de {os.path.basename(ruta)}')
        anios_en_nombre = re.findall(r'(?<!\d)(?:19|20)\d{2}(?!\d)', os.path.basename(ruta))
        if anios_en_nombre and int(anios_en_nombre[0]) != anio:
            raise ValueError(f'{os.path.basename(ruta)} está en la carpeta de {anio} pero su nombre dice {anios_en_nombre[0]}')
        if cod in archivos[anio]:
            raise ValueError(f'Hay dos mapas de {NOMBRES[cod]} para {anio}')
        archivos[anio][cod] = ruta
    faltan = sorted(set(NOMBRES) - set(archivos[anio]))
    if faltan:
        raise ValueError(f'Faltan los mapas de {anio} de: {[NOMBRES[c] for c in faltan]} (carpeta {carpeta})')

for anio in ANIOS:
    print(f'{anio}:')
    for cod, ruta in sorted(archivos[anio].items()):
        print(f'   {cod}  {os.path.basename(ruta)}')
print(f'\nGrilla: {len(grilla):,} celdas')

## Paso 3. Área agropecuaria por celda
Los mapas se leen por franjas de filas para no llenar la memoria. En cada franja, los píxeles con dato pasan a EPSG:9377 y se suman por celda: el área con dato y el área agropecuaria. El área de un píxel se calcula con la latitud de su franja.

In [ ]:
TAM = 2000
AGRO = [9, 21, 35]
FILAS = 256
MINIMO_VALIDO = 0.5

x0g, y0g, x1g, y1g = grilla.total_bounds
CX0, CY0 = int(np.floor(x0g / TAM)), int(np.floor(y0g / TAM))
NX, NY = int(np.ceil(x1g / TAM)) - CX0, int(np.ceil(y1g / TAM)) - CY0

def area_pixel_m2(t, crs, lat):
    if crs.is_geographic:
        return abs(t.a) * 111320 * np.cos(np.radians(lat)) * abs(t.e) * 110574
    return abs(t.a * t.e)

def contar(ruta):
    validos = np.zeros(NX * NY)
    agro = np.zeros(NX * NY)
    fuera = 0.0
    clases = set()
    with rasterio.open(ruta) as src:
        t = src.transform
        sin_dato = src.nodata if src.nodata is not None else 0
        a_9377 = Transformer.from_crs(src.crs.to_wkt(), 'EPSG:9377', always_xy=True)
        for f0 in range(0, src.height, FILAS):
            n = min(FILAS, src.height - f0)
            bloque = src.read(1, window=Window(0, f0, src.width, n))
            ff, cc = np.nonzero(bloque != sin_dato)
            if ff.size == 0:
                continue
            clase = bloque[ff, cc]
            clases.update(np.unique(clase).tolist())
            col = cc + 0.5
            fil = ff + f0 + 0.5
            x, y = a_9377.transform(t.c + t.a * col + t.b * fil, t.f + t.d * col + t.e * fil)
            cx = np.floor(np.asarray(x) / TAM).astype(np.int64) - CX0
            cy = np.floor(np.asarray(y) / TAM).astype(np.int64) - CY0
            dentro = (cx >= 0) & (cx < NX) & (cy >= 0) & (cy < NY)
            a_pix = area_pixel_m2(t, src.crs, t.f + t.e * (f0 + n / 2))
            fuera += (~dentro).sum() * a_pix
            clave = cx[dentro] * NY + cy[dentro]
            validos += np.bincount(clave, minlength=NX * NY) * a_pix
            agro += np.bincount(clave[np.isin(clase[dentro], AGRO)], minlength=NX * NY) * a_pix
    return validos, agro, fuera, clases

claves = np.arange(NX * NY)
ids = pd.Series([f'x{(CX0 + k // NY) * 2}_y{(CY0 + k % NY) * 2}' for k in claves])

tabla = pd.DataFrame({'id_celda': ids})
por_mpio = []
for anio in ANIOS:
    v_total = np.zeros(NX * NY)
    a_total = np.zeros(NX * NY)
    for cod, ruta in sorted(archivos[anio].items()):
        v, a, fuera, clases = contar(ruta)
        v_total += v
        a_total += a
        por_mpio.append({'anio': anio, 'mpio_cdpmp': cod, 'ha_con_dato': v.sum() / 1e4,
                         'ha_agro': a.sum() / 1e4, 'ha_fuera_rectangulo': fuera / 1e4,
                         'clases': sorted(int(c) for c in clases)})
        print(f'{anio} {cod}: {v.sum() / 1e4:,.0f} ha con dato, {a.sum() / 1e4:,.0f} ha agropecuarias')
    tabla[f'ha_con_dato_{anio}'] = v_total / 1e4
    tabla[f'agro_ha_{anio}'] = a_total / 1e4
por_mpio = pd.DataFrame(por_mpio)
tabla = tabla[tabla.filter(like='ha_con_dato').sum(axis=1) > 0].reset_index(drop=True)
print(f'\nCeldas con algún píxel: {len(tabla):,}')

## Paso 4. Unir a la grilla y calcular el % agropecuario
Los píxeles que caen en una celda que no está en la grilla (por pequeñas diferencias entre el borde de los mapas y el de los municipios del DANE) se reportan y no se usan.

In [ ]:
celdas = grilla[['id_celda', 'mpio_cdpmp', 'mpio_cnmbr', 'area_km2', 'geometry']].merge(tabla, on='id_celda', how='left')
sobrantes = tabla[~tabla['id_celda'].isin(grilla['id_celda'])]

for anio in ANIOS:
    con_dato = celdas[f'ha_con_dato_{anio}'].fillna(0)
    celdas[f'ha_con_dato_{anio}'] = con_dato
    celdas[f'agro_ha_{anio}'] = celdas[f'agro_ha_{anio}'].fillna(0)
    fraccion = con_dato / (celdas['area_km2'] * 100)
    celdas[f'frac_valida_{anio}'] = fraccion.clip(upper=1)
    celdas[f'agro_pct_{anio}'] = np.where(con_dato > 0, celdas[f'agro_ha_{anio}'] / con_dato.where(con_dato > 0) * 100, np.nan)
    print(f'{anio}: {(fraccion > 1.05).sum():,} celdas con más de 105 % del área con dato '
          f'| {(con_dato == 0).sum():,} celdas sin dato | fuera de la grilla: {sobrantes[f"ha_con_dato_{anio}"].sum():,.0f} ha')

celdas['cambio_agro_pp'] = celdas['agro_pct_2024'] - celdas['agro_pct_2000']
celdas['celda_valida'] = (celdas['frac_valida_2000'] >= MINIMO_VALIDO) & (celdas['frac_valida_2024'] >= MINIMO_VALIDO)

assert celdas['id_celda'].is_unique and len(celdas) == len(grilla)
print(f'\nCeldas válidas en los dos años: {celdas["celda_valida"].sum():,} de {len(celdas):,}')

## Paso 5. Verificación con las tablas municipales de MapBiomas
Las tablas municipales y los mapas son el mismo producto, así que el % agropecuario de cada municipio calculado con los píxeles debe coincidir casi exactamente con el de las tablas. Una diferencia de más de 1 punto indica que algún mapa es de otra colección o de otro año.

In [ ]:
filas = []
for ruta in sorted(glob.glob(os.path.join(CRUDOS, '*.csv'))):
    cod = re.search(r'(\d{5})_', os.path.basename(ruta)).group(1)
    tabla_mb = pd.read_csv(ruta, encoding='utf-8-sig')
    tabla_mb['Level 2'] = tabla_mb['Level 2'].fillna('').str.strip()
    tabla_mb = tabla_mb[tabla_mb['Level 2'] != '']
    for anio in ANIOS:
        total = pd.to_numeric(tabla_mb[str(anio)], errors='coerce').sum()
        agro = pd.to_numeric(tabla_mb.loc[tabla_mb['Level 1'] == 'Agricultural and livestock area', str(anio)], errors='coerce').sum()
        filas.append({'anio': anio, 'mpio_cdpmp': cod, 'agro_pct_tabla': agro / total * 100, 'ha_tabla': total})
referencia = pd.DataFrame(filas)

control = por_mpio.merge(referencia, on=['anio', 'mpio_cdpmp'])
control['agro_pct_mapa'] = control['ha_agro'] / control['ha_con_dato'] * 100
control['dif_pp'] = control['agro_pct_mapa'] - control['agro_pct_tabla']
control['dif_area_pct'] = (control['ha_con_dato'] / control['ha_tabla'] - 1) * 100
control['municipio'] = control['mpio_cdpmp'].map(dict(zip(mpios['mpio_cdpmp'], mpios['mpio_cnmbr'])))
print(control[['anio', 'municipio', 'agro_pct_tabla', 'agro_pct_mapa', 'dif_pp', 'dif_area_pct']].round(2).to_string(index=False))

desconocidas = sorted(set(sum(por_mpio['clases'], [])) - {3, 6, 9, 11, 12, 13, 21, 23, 24, 25, 29, 30, 31, 33, 35, 68, 75})
print('\nClases que no están en la leyenda usada:', desconocidas or 'ninguna')

region = celdas[[f'agro_ha_{a}' for a in ANIOS] + [f'ha_con_dato_{a}' for a in ANIOS]].sum()
PCT_REGION = {a: region[f'agro_ha_{a}'] / region[f'ha_con_dato_{a}'] * 100 for a in ANIOS}
print('Altillanura (mapas):', ' | '.join(f'{a}: {p:.2f} %' for a, p in PCT_REGION.items()))
grandes = control[control['dif_pp'].abs() > 1]
print('\nVerificación OK' if grandes.empty else f'\nREVISAR: {len(grandes)} municipio-año difieren más de 1 punto de las tablas')

## Paso 6. Mapa del % agropecuario y de su cambio
Solo se colorean las celdas válidas (al menos la mitad del área con dato en los dos años).

In [ ]:
def coma(valor, decimales=1):
    return f'{valor:.{decimales}f}'.replace('.', ',')

validas = celdas[celdas['celda_valida']]
crecen = int((validas['cambio_agro_pp'] > 1).sum())
limite = float(np.nanpercentile(validas['cambio_agro_pp'].abs(), 98)) or 1.0

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
validas.plot(column='agro_pct_2024', cmap='YlOrBr', vmin=0, vmax=100, linewidth=0, ax=axes[0],
             legend=True, legend_kwds={'label': '% del área de la celda (2024)', 'shrink': 0.6})
validas.plot(column='cambio_agro_pp', cmap='RdBu_r', vmin=-limite, vmax=limite, linewidth=0, ax=axes[1],
             legend=True, legend_kwds={'label': 'Cambio 2000–2024 (puntos porcentuales)', 'shrink': 0.6})
for ax in axes:
    mpios.boundary.plot(ax=ax, color='black', linewidth=0.6)
    ax.set_axis_off()
axes[0].set_title('Uso agropecuario en 2024', fontsize=11)
axes[1].set_title('Cambio entre 2000 y 2024', fontsize=11)
n_crecen = f'{crecen:,}'.replace(',', '.')
titulo = (f'El uso agropecuario pasó de {coma(PCT_REGION[2000])} % a {coma(PCT_REGION[2024])} % de la Altillanura '
          f'entre 2000 y 2024 y creció más de 1 punto en {n_crecen} celdas')
fig.suptitle(titulo, fontsize=13)
fig.text(0.01, 0.01, 'Fuente: MapBiomas Colombia, Colección 3 (clases 9, 21 y 35). Celdas de 2 × 2 km en EPSG:9377 (cuaderno 02).',
         fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(GRAFICAS, '30_agro_celda.png'), dpi=150, bbox_inches='tight')
plt.show()

## Paso 7. Guardar la capa agropecuaria
Se guarda en `datos/comunes/agro_celda.csv`, una fila por celda de la grilla, y se vuelve a leer para comprobarla.

In [ ]:
COLUMNAS = ['id_celda', 'agro_pct_2000', 'agro_pct_2024', 'cambio_agro_pp', 'agro_ha_2000', 'agro_ha_2024',
            'frac_valida_2000', 'frac_valida_2024', 'celda_valida']
SALIDA = os.path.join(COMUNES, 'agro_celda.csv')
celdas[COLUMNAS].round(4).to_csv(SALIDA, index=False)

leida = pd.read_csv(SALIDA)
assert len(leida) == len(grilla) and leida['id_celda'].is_unique
assert set(leida['id_celda']) == set(grilla['id_celda'])
print(f'Guardado: {SALIDA} ({len(leida):,} filas)')
leida.describe().round(2)